# Fabric Defect Detector — Training (Google Colab)

Run this notebook on Colab with a GPU runtime (Runtime -> Change runtime type -> GPU).

It downloads the dataset via `kagglehub`, fine-tunes a pretrained CNN, and saves `best_model.pt`,
which you then download and place in the local `models/` folder to power the Streamlit demo.

## 1. Install dependencies

In [ ]:
!pip install -q kagglehub

## 2. Download the dataset

This will prompt for Kaggle credentials (username + API key) the first time.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("ziya07/multi-class-fabric-defect-detection-dataset")
print("Dataset downloaded to:", dataset_path)

import os
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        dirs[:] = []

## 3. Data loading & transforms

Mirrors `src/data_loader.py` — handles both a pre-split (train/val/test) and a flat
(class-subfolders only) dataset layout.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SPLIT_ALIASES = {
    "train": ["train", "training"],
    "val": ["val", "valid", "validation"],
    "test": ["test", "testing"],
}

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

def find_split_dirs(data_dir):
    entries = {e.lower(): e for e in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, e))}
    found = {}
    for split, aliases in SPLIT_ALIASES.items():
        for alias in aliases:
            if alias in entries:
                found[split] = os.path.join(data_dir, entries[alias])
                break
    return found if "train" in found else {}

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

split_dirs = find_split_dirs(dataset_path)
generator = torch.Generator().manual_seed(SEED)

if split_dirs:
    train_ds = datasets.ImageFolder(split_dirs["train"], transform=train_tf)
    class_names = train_ds.classes
    val_ds = datasets.ImageFolder(split_dirs["val"], transform=eval_tf) if "val" in split_dirs else None
    test_ds = datasets.ImageFolder(split_dirs["test"], transform=eval_tf) if "test" in split_dirs else None
    if val_ds is None:
        n_val = int(len(train_ds) * 0.15)
        train_ds, val_ds = random_split(train_ds, [len(train_ds) - n_val, n_val], generator=generator)
    if test_ds is None:
        n_test = int(len(train_ds) * 0.15)
        train_ds, test_ds = random_split(train_ds, [len(train_ds) - n_test, n_test], generator=generator)
else:
    full_ds = datasets.ImageFolder(dataset_path, transform=train_tf)
    class_names = full_ds.classes
    n_total = len(full_ds)
    n_val, n_test = int(n_total * 0.15), int(n_total * 0.15)
    n_train = n_total - n_val - n_test
    train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test], generator=generator)
    eval_full_ds = datasets.ImageFolder(dataset_path, transform=eval_tf)
    val_ds.dataset = eval_full_ds
    test_ds.dataset = eval_full_ds

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train/val/test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

## 4. Model — pretrained backbone with fine-tuned head

In [ ]:
import torch.nn as nn
from torchvision import models

ARCH = "resnet18"  # or "mobilenet_v2"

def build_model(arch, num_classes):
    if arch == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = build_model(ARCH, num_classes=len(class_names)).to(device)

## 5. Training loop

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.notebook import tqdm

EPOCHS = 15
LR = 1e-4
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

best_val_acc = 0.0
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_acc)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "arch": ARCH,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "val_acc": val_acc,
        }, "best_model.pt")
        print(f"  -> new best model saved (val_acc={val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs.")
            break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## 6. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test accuracy: {accuracy:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(max(6, len(class_names)), max(5, len(class_names) * 0.8)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix (test accuracy={accuracy:.4f})")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## 7. Download the trained model

Download `best_model.pt` and place it in the local `models/` folder to power the Streamlit demo.

In [ ]:
from google.colab import files

files.download("best_model.pt")
files.download("confusion_matrix.png")